<a href="https://colab.research.google.com/github/Lokesh66666/CV_workshop/blob/main/Yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

YOLO


In [29]:
# Core libraries
import tensorflow as tf            # Main deep learning library
import numpy as np                # For numerical operations (arrays, math)
import matplotlib.pyplot as plt   # For plotting graphs and images

# Dataset
from tensorflow.keras.datasets import cifar10   # CIFAR-10 image dataset

# Model and layers
from tensorflow.keras import layers, models     # To build neural network models

# Pretrained model
from tensorflow.keras.applications import MobileNetV2   # Pretrained CNN model

# Utilities
from sklearn.metrics import classification_report, confusion_matrix  # Evaluation metrics
import seaborn as sns   # For better visualization (like heatmaps)

Load dataset

In [30]:
# Load CIFAR-10 dataset (automatically split into train and test)
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# x_train → training images
# y_train → labels for training images
# x_test → testing images
# y_test → labels for testing images

# Print shapes to understand the data structure
print("Train shape:", x_train.shape)
# Example: (50000, 32, 32, 3) -> 50k images, each 32x32 with 3 color channels

print("Test shape:", x_test.shape)
# Example: (10000, 32, 32, 3)

Train shape: (50000, 32, 32, 3)
Test shape: (10000, 32, 32, 3)


normalized data

In [31]:
# Normalize pixel values (0–255 -> 0–1)
x_train = x_train / 255.0
x_test = x_test / 255.0


Data augmentation

In [32]:
# Data augmentation helps the model learn better by showing slightly modified images
# This reduces overfitting and improves performance on new data

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),     # Randomly flips images left ↔ right
    layers.RandomRotation(0.1),          # Slightly rotates images (10% range)
    layers.RandomZoom(0.1)               # Slightly zooms in/out
])

load pritrained model


In [33]:
# Load MobileNetV2 model without the final classification layer (top layers)
base_model = MobileNetV2(input_shape=(32,32,3),
                         include_top=False,       # Remove default output layer
                         weights='imagenet')      # Use pretrained weights from ImageNet

# Freeze the base model so its learned features are not changed during training
base_model.trainable = False

/tmp/ipykernel_1847/3232108708.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(input_shape=(32,32,3),


Build Advanced model

In [34]:
# Freeze the base model so its learned features are not changed during training
base_model.trainable = False

# Build Advanced Model
# Build custom model on top of the pretrained base model
model = models.Sequential([
    data_augmentation,              # Apply random transformations to input images
    base_model,                     # Use pretrained MobileNetV2 as feature extractor
    layers.GlobalAveragePooling2D(),# Convert feature maps into a single vector
    layers.BatchNormalization(),    # Normalize features to make training stable
    layers.Dense(128, activation="relu"),  # Fully connected layer for learning patterns
    layers.Dropout(0.5),            # Randomly drop neurons to reduce overfitting
    layers.Dense(10, activation="softmax") # Output layer for 10 classes
])

# Display model architecture
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_4 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

compile model


In [35]:
# Compile the model (configure how it learns)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),  # Optimizer to update weights
    loss='sparse_categorical_crossentropy',  # Loss function for multi-class classification
    metrics=['accuracy']  # Metric to track performance
)

E
arly stopping

In [36]:
# Early stopping stops training when model stops improving (prevents overfitting)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',           # Watch validation loss
    patience=3,                   # Wait for 3 epochs before stopping if no improvement
    restore_best_weights=True     # Restore the best model weights after stopping
)

Train model


In [37]:
# Train the model on training data
history = model.fit(
    x_train, y_train,                 # Training data
    epochs=10,                        # Number of times the model sees the data
    validation_data=(x_test, y_test), # Check performance on test data after each epoch
    callbacks=[early_stop]            # Apply early stopping to avoid overfitting
)

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 84s 49ms/step - accuracy: 0.2325 - loss: 2.0712 - val_accuracy: 0.2855 - val_loss: 2.0612
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 75s 48ms/step - accuracy: 0.2575 - loss: 2.0154 - val_accuracy: 0.2877 - val_loss: 2.0749
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 85s 54ms/step - accuracy: 0.2611 - loss: 2.0062 - val_accuracy: 0.2945 - val_loss: 2.0596
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 82s 52ms/step - accuracy: 0.2663 - loss: 1.9986 - val_accuracy: 0.2923 - val_loss: 2.0624
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 83s 53ms/step - accuracy: 0.2688 - loss: 1.9929 - val_accuracy: 0.3034 - val_loss: 2.0623
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 108s 69ms/step - accuracy: 0.2688 - loss: 1.9917 - val_accuracy: 0.2974 - val_loss: 2.0709


In [ ]:
# Unfreeze top layers of base model
base_model.trainable = True

# Freeze initial layers, train only deeper layers
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train again
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 228s 128ms/step - accuracy: 0.1473 - loss: 3.1008 - val_accuracy: 0.1668 - val_loss: 2.2857
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 206s 130ms/step - accuracy: 0.1930 - loss: 2.4686 - val_accuracy: 0.2106 - val_loss: 2.1088
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 181s 116ms/step - accuracy: 0.2187 - loss: 2.2387 - val_accuracy: 0.3000 - val_loss: 1.9470
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 181s 116ms/step - accuracy: 0.2526 - loss: 2.1019 - val_accuracy: 0.3499 - val_loss: 1.8377
Epoch 5/5
 235/1563 ━━━━━━━━━━━━━━━━━━━━ 2:25 109ms/step - accuracy: 0.2627 - loss: 2.0690

Evalution

In [ ]:
# Predictions
y_pred = np.argmax(model.predict(x_test), axis=1)

# Classification report
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 🔴 YOLO Object Detection

# Install ultralytics
!pip install ultralytics

In [ ]:
# Load YOLO model
from ultralytics import YOLO

yolo_model = YOLO('yolov8n.pt')

In [ ]:
# Run detection on image
results = yolo_model('https://ultralytics.com/images/bus.jpg', save=True)

count detect object


In [ ]:
# Count detected objects
for p in results:
    print("Objects detected:", len(p.boxes))

In [ ]:
upload custom image

In [ ]:
# Upload Custom Image
from google.colab import files
uploaded = files.upload()

for img in uploaded.keys():
    results = yolo.model(img, save=True)
